In [16]:
"""
%pip install qiskit-nature==0.7.2
%pip install qiskit-aer==0.17.2
%pip install qiskit-ibm-runtime==0.47.0
%pip install mapomatic==0.14.0
%pip install numpy==2.4.2
%pip install pyscf==2.12.1
%pip install openpyxl==3.1.5
"""

'\n%pip install qiskit-nature==0.7.2\n%pip install qiskit-aer==0.17.2\n%pip install qiskit-ibm-runtime==0.47.0\n%pip install mapomatic==0.14.0\n%pip install numpy==2.4.2\n%pip install pyscf==2.12.1\n%pip install openpyxl==3.1.5\n'

In [17]:
# Data analysis and visualization imports
import ast
import numpy as np
import pandas as pd
import networkx as nx
from statistics import median
import matplotlib.pyplot as plt

# Multiprocessing imports
import multiprocessing
from concurrent.futures import ProcessPoolExecutor, as_completed

# IBM Runtime specific imports
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import EstimatorV2
from qiskit_ibm_runtime.fake_provider import (FakeMarrakesh,
                                              FakeFez,
                                              FakeKingston,
                                              FakeBoston,
                                              FakePittsburgh,
                                              FakeMiami,
                                              FakeAachen,
                                              FakeBerlin
                                              )
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

# Qiskit Nature and Algorithms specific imports
from qiskit_nature.units import DistanceUnit
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_algorithms import MinimumEigensolverResult
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.circuit.library import UCCSD, HartreeFock
from qiskit_nature.second_q.transformers import ActiveSpaceTransformer

In [18]:
import sys

import qiskit
import qiskit_aer
import qiskit_ibm_runtime

print("Python:", sys.version.split()[0])
print("qiskit:", qiskit.__version__)
print("qiskit-aer:", qiskit_aer.__version__)
print("qiskit-ibm-runtime:", qiskit_ibm_runtime.__version__)

Python: 3.13.15
qiskit: 2.5.2
qiskit-aer: 0.17.2
qiskit-ibm-runtime: 0.47.0


# 1°: Configuring fake providers.

In [19]:
fake_marrakesh = FakeMarrakesh()
fake_fez = FakeFez()
fake_kingston = FakeKingston()
fake_boston = FakeBoston()
fake_pittsburgh = FakePittsburgh()
fake_miami = FakeMiami()
fake_aachen = FakeAachen()
fake_berlin = FakeBerlin()

In [20]:
fake_providers = [fake_marrakesh,
                  fake_fez,
                  fake_kingston,
                  fake_boston,
                  fake_pittsburgh,
                  fake_miami,
                  fake_aachen,
                  fake_berlin
              ]

In [21]:
provider_dict = {provider.backend_name: provider for provider in fake_providers}

# 2°: Setting up the chemical problem for the simulations and QPU selection.

In [22]:
# Creating the molecular geometry of BeH2 with STO-3G as minimal basis set and
# H-Be distance of 1.326 angstrom and 180 degrees.

driver = PySCFDriver(
        atom="""H -1.326, 0.0, 0.0
                Be 0.0, 0.0 0.0
                H 1.326, 0.0, 0.0
             """,
        basis='sto3g',
        charge=0,
        spin=0,
        unit=DistanceUnit.ANGSTROM
)

molecule_problem = driver.run()

In [23]:
# The original space has approximately 6 electrons, 7 space orbitals, and 14
# spin orbitals.
# Here we limit the active space to only 4 electrons and 3 space orbitals and
# work with the reduced molecule_problem

active_space_transformer = ActiveSpaceTransformer(num_electrons=4, num_spatial_orbitals=3)
reduced_molecule_problem = active_space_transformer.transform(molecule_problem)

In [24]:
# Here we calculate the Hamiltonian of the second quantization after reduction
# with CAS.
second_q_hamiltonian = reduced_molecule_problem.second_q_ops()[0]

In [25]:
# Here we use the Jordan Wigner mapping
jordan_wigner_mapper = JordanWignerMapper()
qubit_op = jordan_wigner_mapper.map(second_q_hamiltonian)

## 2.1: Ansatz Circuit Construction

In [26]:
# Here we construct the HF state within the CAS and JW mapping.
hf_initial_state = HartreeFock(
      num_particles=reduced_molecule_problem.num_particles,
      num_spatial_orbitals=reduced_molecule_problem.num_spatial_orbitals,
      qubit_mapper=jordan_wigner_mapper
)

In [27]:
# Here we build the ansatz
ansatz = UCCSD(
          reduced_molecule_problem.num_spatial_orbitals,
          reduced_molecule_problem.num_particles,
          initial_state=hf_initial_state,
          qubit_mapper=jordan_wigner_mapper
)

## 2.2 Transpilation and eingenvalue functions

In [28]:
# This small function was created to generate the ansatz and observables in
# terms of Instruction Set Architecture (ISA) operators.
def transpile_to_isa(backend,  optimization_level=0, seed=42, initial_layout=None):
  target = backend.target

  pm = generate_preset_pass_manager(
       target=target,
       initial_layout=initial_layout,
       optimization_level=optimization_level,
       layout_method='sabre',
       routing_method='sabre',
       seed_transpiler=seed
  )

  ansatz_isa = pm.run(ansatz)
  isa_observables = qubit_op.apply_layout(ansatz_isa.layout)
  return ansatz_isa, isa_observables

In [29]:
# Here we add the energy of active space to the frozen energies of the core
# plus nuclear repulsion.
def interpret_exp_val(exp_val, problem):
    sol = MinimumEigensolverResult()
    sol.eigenvalue = np.real(exp_val)
    return problem.interpret(sol).total_energies[0]

# 3°: Benchmarking of QPU Noise on best mapomatic layout

In [30]:
ranking = pd.read_excel("mapomatic_global_ranking.xlsx")

In [31]:
# =====================================================================
# 1. Preparação dos Dados e Dicionários
# =====================================================================

# Carrega o ranking global e converte a "string gigante" de volta para uma lista
ranking['Mapomatic Best Qubits'] = ranking['Mapomatic Best Qubits'].apply(ast.literal_eval)

In [32]:
# =====================================================================
# 2. Função Isolada do Worker Paralelo (Corrigida)
# =====================================================================
def worker_estimator(backend_name, isa_ansatz, isa_qubit_op, sim_seed):
    """
    Função que roda nos núcleos da CPU/TPU host. Executa o EstimatorV2
    no AerSimulator com uma seed de ruído específica.
    """
    provider = provider_dict[backend_name]
    backend = AerSimulator.from_backend(provider)

    # Instancia o EstimatorV2 acoplado ao modelo de ruído do backend
    estimator = EstimatorV2(mode=backend)
    estimator.options.default_shots = 40_000
    estimator.options.simulator.seed_simulator = sim_seed

    # Inicializa os parâmetros no zero (Estado Hartree-Fock)
    params = np.zeros(isa_ansatz.num_parameters)
    pub = (isa_ansatz, isa_qubit_op, params)

    job = estimator.run([pub])
    result = job.result()[0]

    # Extração ultra-segura do valor esperado (lida com 0-D arrays, 1-D arrays e listas)
    raw_ev = result.data.evs
    raw_ev_float = float(np.asarray(raw_ev).flatten()[0])

    return {
        "Backend Name": backend_name,
        "Simulator Seed": sim_seed,
        "Raw Exp Val": raw_ev_float
    }

In [33]:
# =====================================================================
# 3. Loop Principal de Transpilação e Execução Paralela
# =====================================================================

sim_seeds = range(42, 92) # Seeds de 42 a 91 (Total de 50 execuções por QPU)
all_results = []
futures = []

num_cores = multiprocessing.cpu_count()
print(f"Iniciando cálculo de expectativa de energia no AerSimulator (10k shots).")
print(f"Utilizando {num_cores} núcleos para paralelizar as 400 submissões...\n")

with ProcessPoolExecutor(max_workers=num_cores) as executor:

    # Itera sobre cada linha do seu arquivo do Mapomatic
    for index, row in ranking.iterrows():
        backend_name = row['Backend Name']
        transpile_seed = int(row['Seed Used'])
        best_layout = row['Mapomatic Best Qubits'] # Agora é uma lista verdadeira!

        provider = provider_dict[backend_name]
        backend = AerSimulator.from_backend(provider)

        print(f"[{backend_name}] Transpilando (Seed Topológica: {transpile_seed})...")

        # Transpilação cravada nas especificações ótimas (Topologia + Física)
        pm = generate_preset_pass_manager(
            target=backend.target,
            optimization_level=3,
            initial_layout=best_layout,       # O Layout físico do Mapomatic
            seed_transpiler=transpile_seed,   # A Seed vencedora do Sabre
            layout_method='sabre',
            routing_method='sabre'
        )

        # Gera o circuito e o Hamiltoniano ISA uma única vez por máquina
        isa_ansatz = pm.run(ansatz)
        isa_qubit_op = qubit_op.apply_layout(isa_ansatz.layout)

        # Dispara as 50 execuções de ruído (shots) para a fila paralela
        for s_seed in sim_seeds:
            futures.append(executor.submit(
                worker_estimator, backend_name, isa_ansatz, isa_qubit_op, s_seed
            ))

    print("\nSubmissões enviadas! Aguardando a simulação concorrente...\n")

    # Coleta assíncrona e cálculo químico final
    for future in as_completed(futures):
        res = future.result()

        # Interpreta a energia total usando a sua função matemática do notebook
        raw_val = res["Raw Exp Val"]
        total_energy = interpret_exp_val(raw_val, reduced_molecule_problem)

        res["Total Energy (Hartree)"] = total_energy
        all_results.append(res)

print("Todas as 400 execuções concluídas com sucesso!\n")

Iniciando cálculo de expectativa de energia no AerSimulator (10k shots).
Utilizando 44 núcleos para paralelizar as 400 submissões...

[fake_aachen] Transpilando (Seed Topológica: 52)...
[fake_berlin] Transpilando (Seed Topológica: 182)...
[fake_boston] Transpilando (Seed Topológica: 14)...
[fake_fez] Transpilando (Seed Topológica: 14)...
[fake_kingston] Transpilando (Seed Topológica: 34)...
[fake_marrakesh] Transpilando (Seed Topológica: 17)...
[fake_miami] Transpilando (Seed Topológica: 177)...
[fake_pittsburgh] Transpilando (Seed Topológica: 84)...

Submissões enviadas! Aguardando a simulação concorrente...

Todas as 400 execuções concluídas com sucesso!



In [34]:
pd.DataFrame(all_results)


,Backend Name,Simulator Seed,Raw Exp Val,Total Energy (Hartree)
0,fake_aachen,72,-3.737723,-15.381301
1,fake_berlin,71,-3.610904,-15.254482
2,fake_boston,43,-3.767892,-15.411470
3,fake_boston,65,-3.765842,-15.409419
4,fake_aachen,71,-3.736437,-15.380015
...,...,...,...,...
395,fake_pittsburgh,85,-3.737822,-15.381399
396,fake_pittsburgh,88,-3.738403,-15.381981
397,fake_pittsburgh,89,-3.734968,-15.378546
398,fake_pittsburgh,90,-3.738047,-15.381624


In [35]:
# =====================================================================
# 4. Consolidação, Agregação e Exportação dos Dados Base
# =====================================================================

# 1. Cria o DataFrame com os dados brutos de todas as 400 execuções
df_raw_results = pd.DataFrame(all_results)

# Definição da energia exata
exact_energy = -15.56033

# Calcula o erro quadrático para CADA execução individualmente
df_raw_results['Squared Error'] = (df_raw_results['Total Energy (Hartree)'] - exact_energy) ** 2

# 2. Agrupa pela QPU e calcula a média, a mediana, o MSE e o Desvio Padrão (SD)
df_baseline_energy = df_raw_results.groupby('Backend Name').agg(
    Mean_Energy=('Total Energy (Hartree)', 'mean'),
    Median_Energy=('Total Energy (Hartree)', 'median'),
    MSE=('Squared Error', 'mean'),
    SD=('Total Energy (Hartree)', 'std')
).reset_index()

# Renomeia as colunas para o padrão do artigo
df_baseline_energy.rename(columns={
    'Mean_Energy': 'Mean Total Energy (Hartree)',
    'Median_Energy': 'Median Total Energy (Hartree)'
}, inplace=True)

# Módulo da média da energia menos o valor de referência
df_baseline_energy['Absolute Difference'] = abs(df_baseline_energy['Mean Total Energy (Hartree)'] - exact_energy)

# 4. Ordena o DataFrame por ordem alfabética (Backend Name)
df_baseline_energy = df_baseline_energy.sort_values(by='Backend Name', ascending=True).reset_index(drop=True)

# Formatação visual para manter os números organizados com 6 casas decimais
df_baseline_styled = df_baseline_energy.style.format({
    "Mean Total Energy (Hartree)": "{:.6f}",
    "Median Total Energy (Hartree)": "{:.6f}",
    "MSE": "{:.6f}",
    "SD": "{:.6f}",
    "Absolute Difference": "{:.6f}"
})

display(df_baseline_styled)

# Exporta os dados brutos e os dados agregados para segurança
df_raw_results.to_excel('vqe_baseline_raw_all_400_runs.xlsx', index=False)
df_baseline_energy.to_excel('vqe_baseline_energy_aggregated.xlsx', index=False)


,Backend Name,Mean Total Energy (Hartree),Median Total Energy (Hartree),MSE,SD,Absolute Difference
0,fake_aachen,-15.380771,-15.380625,0.032243,0.001394,0.179559
1,fake_berlin,-15.253513,-15.253680,0.094140,0.001764,0.306817
2,fake_boston,-15.410084,-15.410192,0.022575,0.001198,0.150246
3,fake_fez,-15.196840,-15.196900,0.132128,0.001831,0.363490
4,fake_kingston,-15.348585,-15.348528,0.044839,0.001660,0.211745
5,fake_marrakesh,-15.362945,-15.362888,0.038963,0.001393,0.197385
6,fake_miami,-15.068807,-15.068319,0.241600,0.002401,0.491523
7,fake_pittsburgh,-15.380240,-15.380209,0.032435,0.001640,0.180090
